In [1]:
"""
MATSim population editor: duplicate selected plan to micro_car variant.

- Eligibility: persons with subpopulation == "person" and household_size <= 2.
- If the SELECTED plan contains any <leg mode="car">:
    * Remove "score" and "selected" attributes from the SELECTED plan.
    * Duplicate that plan; in the duplicate:
        - change all 'car' legs to 'micro_car'
        - update 'routingMode'="car" to "micro_car" on those legs
        - update adjacent walk legs' routingMode from "car" to "micro_car"
        - change "car interaction" activities to "micro_car interaction"
    * Append the micro_car plan to the person's plans.
- Robust I/O for .xml and .xml.gz; preserves MATSim DTD.
- Reports detailed statistics.
"""

from lxml import etree
from pathlib import Path
from collections import Counter
import gzip
import copy

# --------------------- CONFIG ---------------------
IN_PATH  = Path("/home/superadmin/Desktop/Shahriar/matsim-berlin/input/v6.4/berlin-v6.4-10pct.plans.xml.gz")
OUT_PATH = Path("/home/superadmin/Desktop/Shahriar/matsim-berlin/input/v6.4/MC_berlin-v6.4-10pct.plans.xml.gz")

MICRO_MODE = "micro_car"
CAR_MODE   = "car"

# MATSim DTD (kept identical to the input’s)
DTD_POPULATION = '<!DOCTYPE population SYSTEM "http://www.matsim.org/files/dtd/population_v6.dtd">'

# --------------------- HELPERS ---------------------
def parse_xml_maybe_gz(path: Path, parser: etree.XMLParser) -> etree._ElementTree:
    if path.suffix == ".gz":
        with gzip.open(path, "rb") as fh:
            return etree.parse(fh, parser)
    return etree.parse(str(path), parser)

def write_xml_maybe_gz(path: Path, xml_bytes: bytes) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".gz":
        with gzip.open(path, "wb") as fh:
            fh.write(xml_bytes)
    else:
        path.write_bytes(xml_bytes)

def get_person_attr(person_el, name, default=None, cast=str):
    attrs_el = person_el.find("attributes")
    if attrs_el is None:
        return default
    for a in attrs_el.findall("attribute"):
        if a.get("name") == name:
            txt = (a.text or "").strip()
            if txt == "":
                return default
            try:
                return cast(txt)
            except Exception:
                return default
    return default

def get_selected_plan(person_el):
    plans = person_el.findall("plan")
    if not plans:
        return None
    for p in plans:
        if p.get("selected", "").lower() == "yes":
            return p
    return plans[0]

def subpopulation_counts(root):
    cnt = Counter()
    total = 0
    for person in root.findall("person"):
        total += 1
        sp = get_person_attr(person, "subpopulation", default="(missing)", cast=str)
        cnt[sp] += 1
    return total, cnt

def plan_has_car_leg(plan_el) -> bool:
    for leg in plan_el.findall("leg"):
        if (leg.get("mode") or "").strip() == CAR_MODE:
            return True
    return False

def strip_plan_score_and_selected(plan_el) -> None:
    if "score" in plan_el.attrib:
        del plan_el.attrib["score"]
    if "selected" in plan_el.attrib:
        del plan_el.attrib["selected"]

def mutate_plan_car_to_micro(plan_el, micro_mode: str) -> int:
    """
    In-place mutate given plan:
      - change all car legs to micro_mode and their routingMode from 'car' to micro_mode
      - update adjacent walk legs' routingMode if set to 'car'
      - change 'car interaction' activities to f'{micro_mode} interaction'
    Returns number of car legs changed.
    """
    changed = 0
    children = list(plan_el)

    def _set_walk_routing_mode(leg_el):
        if leg_el is not None and leg_el.tag == "leg" and (leg_el.get("mode") or "").strip() == "walk":
            attrs = leg_el.find("attributes")
            if attrs is not None:
                for attr in attrs.findall("attribute"):
                    if attr.get("name") == "routingMode" and (attr.text or "").strip() == CAR_MODE:
                        attr.text = micro_mode

    def _upgrade_interaction(activity_el):
        if activity_el is not None and activity_el.tag == "activity":
            if (activity_el.get("type") or "").strip() == "car interaction":
                activity_el.set("type", f"{micro_mode} interaction")

    for idx, leg in enumerate(children):
        if leg.tag != "leg":
            continue
        if (leg.get("mode") or "").strip() != CAR_MODE:
            continue

        # change leg mode
        leg.set("mode", micro_mode)

        # change routingMode inside leg attributes
        attrs_el = leg.find("attributes")
        if attrs_el is not None:
            for attr in attrs_el.findall("attribute"):
                if attr.get("name") == "routingMode" and (attr.text or "").strip() == CAR_MODE:
                    attr.text = micro_mode

        # neighbors for access/egress + interactions
        prev1 = children[idx - 1] if idx - 1 >= 0 else None
        next1 = children[idx + 1] if idx + 1 < len(children) else None
        prev2 = children[idx - 2] if idx - 2 >= 0 else None
        next2 = children[idx + 2] if idx + 2 < len(children) else None

        _upgrade_interaction(prev1)
        _upgrade_interaction(next1)

        if prev1 is not None and prev1.tag == "leg":
            _set_walk_routing_mode(prev1)
        elif prev2 is not None and prev1 is not None and prev1.tag == "activity":
            _set_walk_routing_mode(prev2)

        if next1 is not None and next1.tag == "leg":
            _set_walk_routing_mode(next1)
        elif next2 is not None and next1 is not None and next1.tag == "activity":
            _set_walk_routing_mode(next2)

        changed += 1

    return changed

def pct(n, d):
    return (100.0 * n / d) if d else 0.0

# --------------------- MAIN ---------------------
parser = etree.XMLParser(remove_blank_text=False)
tree = parse_xml_maybe_gz(IN_PATH, parser)
root = tree.getroot()

total_persons_before, subpops_before = subpopulation_counts(root)

eligible_persons = 0
persons_modified = 0
car_legs_in_micro_plans = 0
plans_added = 0
plans_stripped = 0

for person in root.findall("person"):
    subpop = get_person_attr(person, "subpopulation", default=None, cast=str)
    if subpop != "person":
        continue

    hh_size = get_person_attr(person, "household_size", default=None, cast=int)
    if hh_size is None or hh_size > 2:
        continue

    plan = get_selected_plan(person)
    if plan is None:
        continue

    if not plan_has_car_leg(plan):
        continue

    eligible_persons += 1

    # (1) strip score/selected on the original selected plan (car)
    strip_plan_score_and_selected(plan)
    plans_stripped += 1

    # (2) duplicate selected plan and mutate to micro_car
    micro_plan = copy.deepcopy(plan)
    changed = mutate_plan_car_to_micro(micro_plan, MICRO_MODE)
    if changed > 0:
        # ensure duplicated plan also has no score/selected
        strip_plan_score_and_selected(micro_plan)
        # append to person
        person.append(micro_plan)
        persons_modified += 1
        car_legs_in_micro_plans += changed
        plans_added += 1
    # if no car legs changed (should not happen because predicate checked), do not append

total_persons_after, subpops_after = subpopulation_counts(root)

xml_bytes = etree.tostring(
    tree,
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
    doctype=DTD_POPULATION
)
write_xml_maybe_gz(OUT_PATH, xml_bytes)

print("=== MATSim plan duplication to micro_car (no scoring/selection) ===")
print(f"Input file:  {IN_PATH}")
print(f"Output file: {OUT_PATH}\n")

print("Eligibility: subpopulation=='person' and household_size<=2; action applied if SELECTED plan contains any car leg.\n")

print(f"Total persons (before): {total_persons_before}")
for sp, c in subpops_before.items():
    print(f"  - {sp:>18}: {c}")

print(f"\nEligible persons scanned: {eligible_persons}")
print(f"Persons with micro_car duplicate appended: {persons_modified}")
print(f"  Share of total population: {persons_modified} / {total_persons_before} ({pct(persons_modified, total_persons_before):.2f}%)")
print(f"Plans stripped of score/selected: {plans_stripped}")
print(f"Micro_car plans appended:        {plans_added}")
print(f"Total car legs mutated in appended plans: {car_legs_in_micro_plans}")

print("\nTotal persons (after):", total_persons_after)
for sp, c in subpops_after.items():
    print(f"  - {sp:>18}: {c}")


=== MATSim plan duplication to micro_car (no scoring/selection) ===
Input file:  /home/superadmin/Desktop/Shahriar/matsim-berlin/input/v6.4/berlin-v6.4-10pct.plans.xml.gz
Output file: /home/superadmin/Desktop/Shahriar/matsim-berlin/input/v6.4/MC_berlin-v6.4-10pct.plans.xml.gz

Eligibility: subpopulation=='person' and household_size<=2; action applied if SELECTED plan contains any car leg.

Total persons (before): 526111
  -             person: 492874
  - commercialPersonTraffic: 7572
  - commercialPersonTraffic_service: 11645
  -            freight: 5459
  -       goodsTraffic: 8561

Eligible persons scanned: 66693
Persons with micro_car duplicate appended: 66693
  Share of total population: 66693 / 526111 (12.68%)
Plans stripped of score/selected: 66693
Micro_car plans appended:        66693
Total car legs mutated in appended plans: 258654

Total persons (after): 526111
  -             person: 492874
  - commercialPersonTraffic: 7572
  - commercialPersonTraffic_service: 11645
  -     